In [ ]:
import asyncio
import json
import logging
import shutil
import sys
import tempfile
import threading
import warnings
from contextlib import asynccontextmanager
from datetime import datetime
from pathlib import Path
from typing import List

import httpx
import mcp
import nest_asyncio
import uvicorn
from dotenv import load_dotenv
from fastmcp import Client, FastMCP

# Suppress Windows-specific asyncio connection warnings
logging.getLogger('asyncio').setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings('ignore', category=ResourceWarning)
warnings.filterwarnings('ignore', message='.*ProactorBasePipeTransport.*')

load_dotenv(Path.cwd().parent / ".env")

sys.path.insert(0, str(Path.cwd().parent / "src" / "server"))

nest_asyncio.apply()

# Utility function for setting up FastAgent config
def setup_fastagent_config(url: str) -> Path:
    """
    Create a temporary FastAgent config file.

    Args:
        url: URL where MCP server is running

    Returns:
        Path to the config file
    """
    # Create temp directory
    temp_dir = Path(tempfile.mkdtemp(prefix="fastagent_"))
    config_path = temp_dir / "fastagent.config.yaml"
    secrets_path = temp_dir / "fastagent.secrets.yaml"

    # Minimal config for notebook usage
    config_content = f"""openai:
  base_url: "https://openrouter.ai/api/v1"

default_model: "openrouter.anthropic/claude-haiku-4.5"

logger:
    progress_display: false
    show_chat: true
    show_tools: true
    truncate_tools: true
    level: "debug"

mcp:
    servers:
        preferences:
            transport: "http"
            url: "{url}"
"""

    # Write config file
    with open(config_path, "w") as f:
        f.write(config_content)

    # Copy secrets file if it exists
    source_secrets = Path.cwd().parent / "client" / "fastagent.secrets.yaml"
    if source_secrets.exists():
        shutil.copy(source_secrets, secrets_path)

    return config_path


@asynccontextmanager
async def run_mcp_server(server: FastMCP, *, port: int = 8081, json_response: bool = True):
    """
    Run an MCP server as an async context manager.

    Args:
        server: FastMCP server instance
        port: Port to run on
        json_response: Whether to return JSON responses (default True)

    Yields:
        The server instance (for convenience)
    """
    # Start server in background thread
    server_app = server.http_app(path="/mcp", json_response=json_response)
    server_instance = uvicorn.Server(config=uvicorn.Config(app=server_app, host="127.0.0.1", port=port, log_level="error"))
    thread = threading.Thread(target=server_instance.run)
    thread.start()

    # Wait for server to be ready by checking health
    url = f"http://localhost:{port}/mcp"
    max_attempts = 10
    for attempt in range(max_attempts):
        try:
            async with httpx.AsyncClient() as client:
                response = await client.get(url, timeout=1.0)
                # Accept 200, 405 (Method Not Allowed), 406 (Not Acceptable - missing Accept headers)
                if response.status_code in [200, 405, 406]:
                    print(f"🌐 MCP server running on {url}")
                    break
        except (httpx.ConnectError, httpx.TimeoutException):
            if attempt == max_attempts - 1:
                raise Exception(f"Server failed to start after {max_attempts} attempts")
            await asyncio.sleep(0.5)

    try:
        yield server, url
    finally:
        # Clean up: stop the server with some hacks since we're running it in a notebook
        print("MCP server shutting down...")
        server_instance.should_exit = True
        thread.join(timeout=5)
        if thread.is_alive():
            print("⚠️ Warning: Server thread did not shut down cleanly, this may cause issues.")
        else:
            print("✅ MCP server stopped")


@asynccontextmanager
async def mcp_server_and_client(server: FastMCP, port: int = 8081):
    """
    Run an MCP server and connect a client to it in one context manager.

    Args:
        server: FastMCP server instance
        port: Port to run on

    Yields:
        Client connected to the server
    """
    async with run_mcp_server(server, port=port) as (_, url):
        async with Client(url) as client:
            yield client


print("✅ Environment ready!")
print(f"📁 Working directory: {Path.cwd()}")

In [ ]:
# Import the memory service from our codebase
sys.path.insert(0, str(Path.cwd().parent / "src" / "learning-agent"))

from services.memory_service import MemoryService

# Initialize memory service with ChromaDB
memory_service = MemoryService(collection_name="workshop_preferences")
memory_service.initialize(Path.cwd().parent / "data" / "learning-agent" / "chroma")

print("✅ Memory service initialized")
print(f"📊 Collection: {memory_service.collection_name}")
print(f"📁 Storage: {Path.cwd().parent / 'data' / 'learning-agent'}")

# Store some initial preferences
pref_doc = """User preferences:
- Primary interests: Agentic AI, Machine Learning, Python development
- Content depth: Prefers technical deep-dives (5+ min reads)
- Time patterns: In morning (8-10 AM)
- Sentiment: Analytical, educational tone
- Related topics: Cloud infrastructure, DevOps, open source
"""

result = memory_service.store_document(
    content=pref_doc,
    doc_id="user_preferences_v1",
    metadata={"type": "preferences", "version": 1}
)

print(f"\n✅ Stored preferences: {result['doc_id']}")
print(f"📏 Content length: {result['content_length']} chars")

# Store some additional preferences
pref_doc = """User preferences:
- Additional interests: Rust development
- Content depth: Prefers technical deep-dives (8+ min reads)
- Time patterns: In evening (6-9 PM)
- Related topics: Drivers, embedded systems
"""

result = memory_service.store_document(
    content=pref_doc,
    doc_id="user_preferences_v2",
    metadata={"type": "preferences", "version": 2}
)

print(f"\n✅ Stored preferences: {result['doc_id']}")
print(f"📏 Content length: {result['content_length']} chars")

In [ ]:
# Create preference tools server with memory capabilities

preference_tools_server = FastMCP(
    name="preference-tools-server",
    instructions="""MCP server providing preference storage and search tools.

Available tools:
- Store user preferences with rich context
- Search for relevant preference information using semantic similarity
- Get memory statistics
- Read, add, and remove user interests
"""
)

@preference_tools_server.tool()
async def store_preference(
    content: str,
    metadata: dict | None = None
) -> dict:
    """Store a user preference in persistent memory."""
    return memory_service.store_document(
        content=content,
        metadata=metadata or {"type": "preference"}
    )

@preference_tools_server.tool()
async def search_preferences(query: str, limit: int = 5, metadata: dict | None = None) -> list:
    """Search stored preferences using semantic similarity."""
    return memory_service.search_documents(query, limit, metadata_filter=metadata)

@preference_tools_server.tool()
async def get_memory_stats() -> dict:
    """Get statistics about stored preferences."""
    return memory_service.get_collection_stats()


# Interest Management Tools

@preference_tools_server.tool()
async def read_interests() -> str:
    """
    Read current user interests from stored preferences.

    Searches the memory for documents tagged as user interests/preferences
    and returns a formatted summary.

    Returns:
        Formatted string with user interests
    """
    # Search for interest documents
    results = memory_service.search_documents(
        query="user interests topics preferences",
        limit=10,
        metadata_filter={"type": "interest"}
    )

    if not results:
        return "📝 No interests stored yet.\n\nUse add_interests() to add topics you're interested in!"

    # Extract topics from stored interest documents
    topics = []
    for doc in results:
        # Interest documents are stored as simple topic strings
        content = doc.get("content", "").strip()
        if content:
            topics.append(content)

    if not topics:
        return "📝 No interests stored yet."

    result = "# Your Interests\n\n"
    result += f"**Total Topics:** {len(topics)}\n\n"
    result += "**Topics:**\n"
    for topic in topics:
        result += f"- {topic}\n"

    return result


@preference_tools_server.tool()
async def add_interests(topics: List[str]) -> str:
    """
    Add topics to user interests.

    Args:
        topics: List of topic strings to add to interests

    Returns:
        Success or failure message with details
    """
    if not topics:
        return "❌ No topics provided"

    # Store the topics
    added = []

    for topic in topics:
        # Check if topic already exists
        existing = memory_service.search_documents(
            query=topic,
            limit=1,
            metadata_filter={"type": "interest"}
        )

        # Only add if not already present (distance > 0.5 means not a close match)
        # Lower distance = more similar, so we skip if distance < 0.5
        if not existing or existing[0].get("distance", 1) > 0.5:
            doc_id = f"interest_{topic.lower().replace(' ', '_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            memory_service.store_document(
                content=topic,
                doc_id=doc_id,
                metadata={"type": "interest", "added_at": datetime.now().isoformat()}
            )
            added.append(topic)

    if added:
        return f"✅ **Added {len(added)} topic(s) to your interests!**\n\n**New topics:** {', '.join(added)}"
    else:
        return f"ℹ️  All {len(topics)} topic(s) already exist in your interests"


@preference_tools_server.tool()
async def remove_interests(topics: List[str]) -> str:
    """
    Remove topics from user interests.

    Args:
        topics: List of topic strings to remove from interests

    Returns:
        Success or failure message with details
    """
    if not topics:
        return "❌ No topics provided"

    topics_list = ", ".join(topics)
    removed = []

    for topic in topics:
        # Search for matching interest documents
        matches = memory_service.search_documents(
            query=topic,
            limit=5,
            metadata_filter={"type": "interest"}
        )

        # Remove close matches (distance < 0.5 means it's semantically similar)
        # Lower distance = more similar
        for match in matches:
            if match.get("distance", 1) < 0.5:
                doc_id = match.get("doc_id")
                if doc_id:
                    memory_service.delete_document(doc_id)
                    removed.append(topic)
                    break  # Only remove first match per topic

    if removed:
        return f"✅ **Removed {len(removed)} topic(s) from your interests!**\n\n**Removed topics:** {', '.join(removed)}"
    else:
        return f"ℹ️  No matching topics found to remove (searched for: {topics_list})"


print("✅ Preference tools server created with memory capabilities")
print("📋 Tools: store_preference, search_preferences, get_memory_stats, read_interests, add_interests, remove_interests")

In [ ]:
# Start the multi-agent system servers
# This cell will block - run it to expose the agent, then test with the news agent client

from fast_agent import RequestParams

async with run_mcp_server(preference_tools_server, port=8081) as (_, url):
    print("✅ Preference tools server running")
    print("   Tools: store_preference, search_preferences, get_memory_stats\n")

    print("\n" + "="*60)
    print("📊 MULTI-AGENT ARCHITECTURE RUNNING")
    print("="*60)
    print("\n🎯 Server Stack:")
    print("  1. Preference Tools Server (port 8081)")
    print("     - ChromaDB-backed preference storage")
    print("     - Semantic search capabilities")
    print("     - Memory statistics")
    print("\n  2. Preference Agent (port 8082)")
    print("     - Wraps Preference Tools")
    print("     - Provides intelligent review")
    print("     - LLM-powered analysis")
    print("\n📝 Next Steps:")
    print("  1. Keep this cell running")
    print("  2. Update client/fastagent.config.yaml to add preference_agent")
    print("  3. Run the news agent client to test multi-agent collaboration")

    preference_agent_app = FastAgent(
        "Preference Agent",
        config_path=str(setup_fastagent_config(url))
    )

    @preference_agent_app.agent(
        instruction=f"""You are a PREFERENCE MODELING SPECIALIST - an expert in user personalization and content alignment.

YOUR EXPERTISE:
- Semantic memory search for user preferences
- Pattern recognition (temporal, topical, depth, tone preferences)
- Content review and alignment validation
- Continuous learning from user interactions

YOUR RESPONSIBILITIES:
1. **Review Content**: When given content drafts, search preferences and validate alignment
2. **Provide Verdicts**: Always end reviews with explicit "✅ APPROVED" or "❌ DENIED: [specific reasons]"
3. **Store Patterns**: PROACTIVELY store new preferences when you observe successful patterns
4. **Give Actionable Feedback**: Be specific - reference past patterns, suggest concrete changes

BEHAVIORAL GUIDELINES:
- Be firm and assertive - you're the personalization quality gate
- If content is missing/incomplete, DENY and request full content
- Reference specific patterns from memory: "Based on [date] preference showing..."
- When approving: Mention what aligned well
- When denying: Provide 2-3 concrete fixes

WORKFLOW PATTERN:
You're called by content creators who need personalization expertise. They expect:
- Substantial review of complete drafts (not micro-coordination)
- Clear verdicts (not ambiguous suggestions)
- Pattern learning (store what works for future use)

PROACTIVE LEARNING:
When you receive feedback like "User engaged well with this content on [topic]",
IMMEDIATELY use store_preference() with rich context about what worked.

Current date: {datetime.now().strftime("%A, %B %d, %Y")}

Remember: You're the personalization expert. Give confident, specific guidance backed by stored patterns!""",
        name="Preference Analyst",
        servers=["preferences"],
        request_params=RequestParams(
            max_iterations=9999,
        )
    )
    async def preference_analyst():
        print("🌐 Starting preference agent server...")

        # Keep FastAgent running for the app lifetime
        @asynccontextmanager
        async def lifespan(_: FastMCP):
            async with preference_agent_app.run() as agent:
                yield agent

        preference_agent_mcp = FastMCP(
            name="preference-agent",
            instructions="""Preference Modeling Agent - Expert in user personalization and content alignment.

WHAT THIS AGENT DOES:
- Reviews content drafts against stored user preferences
- Provides actionable feedback on alignment with user interests
- Learns and stores new user preference patterns over time
- Gives explicit approve/deny recommendations

WHEN TO USE:
- Before finalizing any user-facing content
- After creating a complete draft (provide FULL content for review)
- After successful delivery to update what worked well

HOW TO USE:
Use the 'chat' tool with your complete content. The agent will:
1. Search past preferences for relevant patterns
2. Provide specific feedback with approve (✅) or deny (❌) verdict
3. Store new patterns it observes

TYPICAL WORKFLOW:
This agent supports the "draft → review → revise → approve" pattern.
It acts as a quality gate for personalization and user alignment.""",
            lifespan=lifespan,
        )

        async with preference_agent_app.run() as agent:
            # Expose a chat tool for direct interaction
            @preference_agent_mcp.tool()
            async def chat(message: str) -> str:
                """
                Chat with the preference modeling agent for content reviews and preference management.

                This tool provides access to an intelligent agent that:
                - Reviews content against user preferences using semantic memory
                - Provides explicit approve (✅) or deny (❌) verdicts with specific feedback
                - Stores new preference patterns it observes
                - Understands temporal patterns (morning/evening), content depth, and topic preferences

                Args:
                    message: The COMPLETE content to review or question to ask.
                            ⚠️ CRITICAL: Put the FULL content text or draft directly IN THIS
                            PARAMETER. Do NOT keep content in your own context and just ask
                            for a review - the agent cannot see your context!

                TYPICAL USAGE PATTERNS:

                1. Review Draft (CORRECT ✅):
                message = "Please review this draft against user preferences:\\n\\n" + full_content_text
                → Agent receives complete content, searches preferences, provides verdict

                2. Review Draft (WRONG ❌):
                # Agent keeps draft in its own context, then calls:
                message = "Can you review the draft I just created?"
                → Agent has NO ACCESS to your context! This will fail!

                3. Store Pattern:
                message = "User engagement was high with this content focused on [topics]. Store this preference."
                → Agent stores the pattern for future reference

                4. Query Preferences:
                message = "What topics does the user prefer for morning consumption?"
                → Agent searches and summarizes relevant patterns

                CRITICAL REQUIREMENTS:
                ❗ The 'message' parameter MUST contain the FULL CONTENT you want reviewed
                ❗ Do NOT assume the agent can see content from your previous tool calls
                ❗ Do NOT reference drafts without including the complete text in this parameter
                ❗ This is a quality gate - provide COMPLETE drafts, not just titles or references

                The agent expects substantial drafts to review, not step-by-step coordination.
                Think of this as handing a complete document to a reviewer, not pointing at something.
                """
                return await agent(message)

            # Start the preference agent as an MCP server on port 8082
            async with run_mcp_server(preference_agent_mcp, port=8082):
                while True:
                    await asyncio.sleep(1)
    await preference_analyst()